# Node 02 · 3D 权威分子构造
### 化学身份 → 空间结构身份

调度 `core/node02_master_build.py`。为 finalize 表每行构造【一个权威 3D 主分子】(master SDF)，
强制核心不变量：

```
CanonicalIsoSMILES(built master SDF) == CanonicalIsoSMILES(input isomeric_smiles)
```

即 3D 化只加坐标、不改身份（含立体/电荷）。**不做化学判断**——已由 Node 01 冻结。

**输入**：`V3_microstate_table_final.csv`（Node 01 finalize v0.2.1）
行数与来源由 Cell 2 对照 `Node01_finalize_manifest.json` 断言核验，不写死数字。

**输出**：每分子 `sdf/<name>.sdf` + `identity_manifests.jsonl` + held / failures / orphaned_parents + manifest

**策略**：
- `downstream_eligibility != READY` 不构造 —— 尊重 Node 01 的资格裁定（`HOLD_STEREO` / `HOLD_CHARGE_REVIEW`），Node 02 不重新解释标志
- 几何不可行 → `held`（不静默丢；`maxIterations=0` 即 RDKit 自动 ≈ 10×原子数，故失败者为真不可行）
- 不变量 / 电荷不符 → `QC_FAIL`（记录，不静默通过）
- 孤儿母体审计只报**意外**掉出；有意保留者单独计入 `n_intentionally_held`

**⚠ DRY_RUN**：v0.1.4 起 `dry_run` 默认 `True`（fail-safe），且**必须显式传入** ——
Cell 3 / Cell 4 均需 `N2.run_node02(..., dry_run=DRY_RUN)`，否则不写任何 SDF。
`write_outputs` 带一致性守卫：构建阶段若为 dry-run，则拒绝写出声称已构建的 manifest。

In [2]:
# ── Cell 1 · 配置 ──────────────────────────────────────
from pathlib import Path
import sys
DRY_RUN = False
PROJECT_ROOT = Path("/mnt/hpc/home/wanghongmei/lang2").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
INPUT_CSV = PROJECT_ROOT / "results" / "01_microstates" / "V3_microstate_table_final.csv"
OUT_DIR   = PROJECT_ROOT / "results" / "02_master_molecules"
print("DRY_RUN   =", DRY_RUN)
print("INPUT_CSV =", INPUT_CSV, " 存在:", INPUT_CSV.is_file())
print("OUT_DIR   =", OUT_DIR)

DRY_RUN   = False
INPUT_CSV = /mnt/hpc/home/wanghongmei/lang2/results/01_microstates/V3_microstate_table_final.csv  存在: True
OUT_DIR   = /mnt/hpc/home/wanghongmei/lang2/results/02_master_molecules


In [3]:
# ── Cell 2 · 导入 + 读入 ───────────────────────────────
import importlib, json
from core import master_molecule as mm
from core import node02_master_build as N2
importlib.reload(mm); importlib.reload(N2)
import pandas as pd
from rdkit import Chem

print("node02 版本:", N2.NODE02_VERSION, "| master_molecule:", mm.MASTER_MODULE_VERSION)

df = pd.read_csv(INPUT_CSV, dtype=str)

# 对照 Node 01 finalize manifest 核验行数与来源（不写死数字，链变了会立刻报错）
MANIFEST = INPUT_CSV.parent / "Node01_finalize_manifest.json"
if MANIFEST.is_file():
    fm = json.loads(MANIFEST.read_text(encoding="utf-8"))
    expected = fm["stats"]["n_final_rows"]
    print(f"读入 {len(df)} 行  (finalize manifest 声明 {expected})")
    assert len(df) == expected, (
        f"行数不符：CSV {len(df)} vs manifest {expected} —— "
        f"可能读到了陈旧的 V3_microstate_table_final.csv，请重跑 Node 01 finalize")
    print(f"Node 01 来源: finalize v{fm['version']} | "
          f"input_sha256 {fm['input_sha256'][:12]}…")
    print(f"资格分布: {fm['stats']['downstream_eligibility_counts']}")
else:
    print(f"读入 {len(df)} 行  [警告] 未找到 {MANIFEST.name}，跳过链路核验")

print("\nstereo_status 分布:")
print(df["stereo_status"].value_counts().to_string())

node02 版本: 0.1.4 | master_molecule: 0.1.0
读入 1790 行  (finalize manifest 声明 1790)
Node 01 来源: finalize v0.2.1 | input_sha256 b7abb39bf4da…
资格分布: {'READY': 1785, 'HOLD_STEREO': 4, 'HOLD_CHARGE_REVIEW': 1}

stereo_status 分布:
stereo_status
defined_or_none                    1504
auto_enumerated                     279
held_high_stereo                      4
no_real_stereo_after_correction       3


In [4]:
# ── Cell 3 · 冒烟测试（前若干行，验证不变量）────────────
# 先小规模验证不变量与分类，再决定全量。全量 ~2030 行约 40+ 分钟（3D 嵌入慢）。
SMOKE_N = 30
# manifests, held, failures, stats = N2.run_node02(df.head(SMOKE_N), mm, Chem, OUT_DIR)

manifests, held, failures, stats = N2.run_node02(df.head(SMOKE_N), mm, Chem, OUT_DIR, dry_run=DRY_RUN)

print(f"冒烟 {SMOKE_N} 行:")
print(f"  built={stats['n_built']}  held={stats['n_held']}  qc_fail={stats['n_qc_fail']}")
if failures:
    print("  ⚠ QC失败:")
    for f in failures[:5]:
        print(f"    {f['name']} @ {f.get('stage')}: {str(f.get('error'))[:50]}")
else:
    print("  ✅ 无 QC 失败——不变量成立")

[node02] dry_run=False -> 将写 SDF 到 /mnt/hpc/home/wanghongmei/lang2/results/02_master_molecules/sdf
冒烟 30 行:
  built=30  held=0  qc_fail=0
  ✅ 无 QC 失败——不变量成立


In [5]:
# ── Cell 4 · 全量构造（较慢；建议放后台或分批）──────────
# 注意：3D 嵌入 CPU 密集，全量 ~2030 行约 40+ 分钟。
# 几何不可行的立体异构体约 5s 各判定为 held(不静默丢)。
RUN_FULL = True   # 确认冒烟无误后设 True
if RUN_FULL:
    #manifests, held, failures, stats = N2.run_node02(df, mm, Chem, OUT_DIR)
    manifests, held, failures, stats = N2.run_node02(df, mm, Chem, OUT_DIR, dry_run=DRY_RUN)
    print(f"全量: built={stats['n_built']}  held={stats['n_held']}  qc_fail={stats['n_qc_fail']}")
    print(f"\nheld 原因(三分类): {stats['held_reasons']}")
    print(f"优化方法分布:      {stats['optimization_methods']}")
    # 孤儿母体审计：所有子代都未建成的母体（整体掉出 docking，必须显式报告）
    orph = stats['orphaned_parents']
    print(f"\n孤儿母体审计: {orph}")
    if orph['n_orphaned_parents'] > 0:
        print("  ⚠ 以下母体全部子代均未构建（整体掉出）:")
        for o in stats['orphan_rows'][:15]:
            print(f"    {o['parent_id']}: {o['n_children']}子代全未建 (enumerated={o['enumerated']})")
        print("  → 见 node02_orphaned_parents.csv；需人工判断这些分子是否重要")
    else:
        print("  ✅ 无孤儿母体：每个母体至少有一个子代成功构建")
else:
    print("[未运行全量] 确认冒烟无误后设 RUN_FULL=True")

[node02] dry_run=False -> 将写 SDF 到 /mnt/hpc/home/wanghongmei/lang2/results/02_master_molecules/sdf
全量: built=1765  held=25  qc_fail=0

held 原因(三分类): {'stereo_geometry_infeasible': 20, 'node01_eligibility:HOLD_STEREO': 4, 'node01_eligibility:HOLD_CHARGE_REVIEW': 1}
优化方法分布:      {'MMFF94': 1764, 'none_ETKDG_only': 1}

孤儿母体审计: {'n_orphaned_parents': 0, 'n_orphaned_enumerated': 0, 'n_orphaned_single': 0, 'orphaned_molecules_lost': 0, 'n_intentionally_held': 5, 'note': 'intentionally held parents are NOT counted as orphans'}
  ✅ 无孤儿母体：每个母体至少有一个子代成功构建


In [12]:
importlib.reload(N2)
N2.annotate_refined_from_disk(manifests, Chem)   # 补 refined_stereocentres 进现有产物
manifest, paths = N2.write_outputs(manifests, held, failures, stats, OUT_DIR, INPUT_CSV, dry_run=False)

In [14]:
n = N2.annotate_refined_from_disk(manifests, Chem)
print("backfilled:", n, "| refined_stereo_summary:", stats := stats)  # 见下

backfilled: 0 | refined_stereo_summary: {'n_processed': 1790, 'n_built': 1765, 'n_held': 25, 'n_qc_fail': 0, 'optimization_methods': {'MMFF94': 1764, 'none_ETKDG_only': 1}, 'held_reasons': {'stereo_geometry_infeasible': 20, 'node01_eligibility:HOLD_STEREO': 4, 'node01_eligibility:HOLD_CHARGE_REVIEW': 1}, 'orphaned_parents': {'n_orphaned_parents': 0, 'n_orphaned_enumerated': 0, 'n_orphaned_single': 0, 'orphaned_molecules_lost': 0, 'n_intentionally_held': 5, 'note': 'intentionally held parents are NOT counted as orphans'}, 'dry_run_no_sdf_written': False, 'failures': [], 'held_examples': [{'name': 'pool_mol_117_s01', 'parent_id': 'pool_mol_117', 'reason': 'stereo_geometry_infeasible', 'isomeric_smiles': 'CNc1cncc(C(=O)N2C[C@@H](Nc3c(C(=O)N4C[C@@H]5[C@@H](C4)C5(C)C)cc(Br)cc3[N+](=O)[O-])C3(CC3)C2)c1'}, {'name': 'pool_mol_117_s03', 'parent_id': 'pool_mol_117', 'reason': 'stereo_geometry_infeasible', 'isomeric_smiles': 'CNc1cncc(C(=O)N2C[C@@H](Nc3c(C(=O)N4C[C@H]5[C@H](C4)C5(C)C)cc(Br)cc3[N+

In [16]:
stats["refined_stereo_summary"] = N2._summarize_refined(manifests)
manifest, paths = N2.write_outputs(manifests, held, failures, stats, OUT_DIR, INPUT_CSV, dry_run=False)
print(manifest["stats"]["refined_stereo_summary"])
# 预期: {'n_molecules_with_refined': 209, 'centres_by_class': {'labile_N': 178, 'symmetric_C': 113}, ...}

{'n_molecules_with_refined': 209, 'centres_by_class': {'labile_N': 178, 'symmetric_C': 113}, 'labile_N_enumeration_extra_molecules': 194, 'note': 'refined = microstate-unspecified stereocentres fixed by ETKDG; labile_N are physically interconverting (candidate for epimer expansion); symmetric_C are the same molecule (record only)'}


In [17]:
import json
recs = [json.loads(l) for l in open(OUT_DIR/"identity_manifests.jsonl", encoding="utf-8")]
print("含 refined 字段的记录:", sum("refined_stereocentres" in r for r in recs), "/", len(recs))
print("示例:", next(r["refined_stereocentres"] for r in recs if r["refined_stereocentres"]))

含 refined 字段的记录: 1765 / 1765
示例: [{'atom_idx': 20, 'symbol': 'N', 'class': 'labile_N'}]


In [18]:
# ── Cell 5 · 写盘（仅 DRY_RUN=False 且已构造）───────────
manifest, paths = N2.write_outputs(manifests, held, failures, stats,
                                   OUT_DIR, INPUT_CSV, dry_run=DRY_RUN)
if not DRY_RUN:
    print("✓ 已写:")
    for k,p in paths.items(): print(f"  {k}: {p}")
    print(f"  SDF 目录: {OUT_DIR}/sdf/")
    print("\n✅ Node 02 完成。下一步：Node 03 — 质子化锁定 PDBQT + 正向原子映射。")
else:
    import json
    print("[DRY_RUN] manifest 预览:\n")
    print(json.dumps(manifest, indent=2, ensure_ascii=False)[:1200])

✓ 已写:
  identity_manifests: /mnt/hpc/home/wanghongmei/lang2/results/02_master_molecules/identity_manifests.jsonl
  held: /mnt/hpc/home/wanghongmei/lang2/results/02_master_molecules/node02_held.csv
  orphaned_parents: /mnt/hpc/home/wanghongmei/lang2/results/02_master_molecules/node02_orphaned_parents.csv
  failures: /mnt/hpc/home/wanghongmei/lang2/results/02_master_molecules/node02_qc_failures.json
  manifest: /mnt/hpc/home/wanghongmei/lang2/results/02_master_molecules/Node02_manifest.json
  SDF 目录: /mnt/hpc/home/wanghongmei/lang2/results/02_master_molecules/sdf/

✅ Node 02 完成。下一步：Node 03 — 质子化锁定 PDBQT + 正向原子映射。


In [19]:
# ── Cell 6 v2 · Node 02 落盘后独立复核（读取端 + 从 3D 判立体）──
from pathlib import Path
import json, hashlib, collections
from rdkit import Chem
from core import master_molecule as mm     # 唯一权威规范化算子（会剥离 atom map）

OUT_DIR = Path("/mnt/hpc/home/wanghongmei/lang2/results/02_master_molecules")
SDF_DIR, MANI = OUT_DIR / "sdf", OUT_DIR / "identity_manifests.jsonl"
recs = [json.loads(l) for l in MANI.read_text(encoding="utf-8").splitlines() if l.strip()]
print("records:", len(recs), "| sdf files:", len(list(SDF_DIR.glob("*.sdf"))))

bad_id, bad_chg, bad_hash, bad_map, no3d, unreadable = [], [], [], [], [], []
for r in recs:
    p = Path(r["sdf_path"])
    if hashlib.sha256(p.read_bytes()).hexdigest() != r["sdf_sha256"]:
        bad_hash.append(r["name"])                       # 文件在写出后被改动过
    m = Chem.MolFromMolFile(str(p), sanitize=True, removeHs=False)
    if m is None:
        unreadable.append(r["name"]); continue
    if m.GetNumConformers() == 0 or not m.GetConformer().Is3D():
        no3d.append(r["name"])
    else:
        Chem.AssignStereochemistryFrom3D(m)              # ← 关键：立体由坐标判定，而非标签
    if Chem.GetFormalCharge(m) != r["built_charge"]:
        bad_chg.append((r["name"], r["built_charge"], Chem.GetFormalCharge(m)))
    if mm.canonical_iso_smiles(m) != r["canonical_isomeric_smiles"]:
        bad_id.append((r["name"], r["canonical_isomeric_smiles"], mm.canonical_iso_smiles(m)))
    maps = sorted(a.GetAtomMapNum() for a in m.GetAtoms() if a.GetAtomicNum() > 1)
    if maps != list(range(1, len(maps) + 1)):            # Node 03 正向映射的前提
        bad_map.append(r["name"])

print(f"unreadable={len(unreadable)} no3D={len(no3d)} sdf_hash_drift={len(bad_hash)} "
      f"identity_mismatch={len(bad_id)} charge_mismatch={len(bad_chg)} atommap_broken={len(bad_map)}")
for t in bad_id[:5]:  print("  ID :", t)
for t in bad_chg[:5]: print("  CHG:", t)

h = hashlib.sha256()
for f in sorted(SDF_DIR.glob("*.sdf")):
    h.update(f.name.encode()); h.update(hashlib.sha256(f.read_bytes()).digest())
print("SDF set sha256:", h.hexdigest())   # 上次记录: 0916208006fe5154…

records: 1765 | sdf files: 1765
unreadable=0 no3D=0 sdf_hash_drift=0 identity_mismatch=183 charge_mismatch=0 atommap_broken=0
  ID : ('pool_mol_096', 'CC1CCN(c2cncc(-c3cncc(N4CCC[C@H]([NH+]5CCO[C@@H](C)C5)C4)c3)c2)CC1', 'CC1CCN(c2cncc(-c3cncc(N4CCC[C@H]([N@H+]5CCO[C@@H](C)C5)C4)c3)c2)CC1')
  ID : ('pool_mol_116', 'C[C@H]([C@H](C)c1cncc(-c2cncc([C@@H](C)[NH+]3CCNC(=O)C3(C)C)c2)c1)[C@@H](C)[NH+]1CCNC(=O)C1(C)C', 'C[C@H]([C@H](C)c1cncc(-c2cncc([C@@H](C)[N@H+]3CCNC(=O)C3(C)C)c2)c1)[C@@H](C)[N@H+]1CCNC(=O)C1(C)C')
  ID : ('pool_mol_154', 'CNc1cncc(-c2cncc(N3CCC[C@H]([C@@H]4CCOC[C@@H]4[NH+]4CCC[C@H]([SH](=O)=O)C4)C3)c2)c1', 'CNc1cncc(-c2cncc(N3CCC[C@H]([C@@H]4CCOC[C@@H]4[N@@H+]4CCC[C@H]([SH](=O)=O)C4)C3)c2)c1')
  ID : ('pool_mol_186_s01', 'CC(C)[C@@H](C(=O)N[C@@H]1CCCN([N+](=O)[O-])C1)C(=O)[C@@H]1CCC[NH+](C(=O)[C@@H]2C[NH2+]C[C@H](C)O2)C1', 'CC(C)[C@@H](C(=O)N[C@@H]1CCCN([N+](=O)[O-])C1)C(=O)[C@@H]1CCC[N@@H+](C(=O)[C@@H]2C[NH2+]C[C@H](C)O2)C1')
  ID : ('pool_mol_186_s02', 'CC(C)[C@H](C(=O)N[

In [20]:
# ── Cell 6 v3 · 从坐标核验立体化学（区分"精化"与"翻转"）──
from pathlib import Path
import json, collections
from rdkit import Chem
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

OUT_DIR = Path("/mnt/hpc/home/wanghongmei/lang2/results/02_master_molecules")
recs = [json.loads(l) for l in (OUT_DIR / "identity_manifests.jsonl")
        .read_text(encoding="utf-8").splitlines() if l.strip()]

def _cip(mol):
    m = Chem.Mol(mol); Chem.AssignStereochemistry(m, cleanIt=True, force=True); return m

def check(ref_smi, mol3d):
    """已指定立体中心必须 CIP 一致；未指定者允许获得几何构型（记为 refined）。"""
    ref = _cip(Chem.MolFromSmiles(ref_smi))
    b = Chem.RemoveHs(Chem.Mol(mol3d))
    for a in b.GetAtoms(): a.SetAtomMapNum(0)
    b = _cip(b)
    if Chem.MolToSmiles(ref, isomericSmiles=False) != Chem.MolToSmiles(b, isomericSmiles=False):
        return "constitution_mismatch", 0
    match = b.GetSubstructMatch(ref, useChirality=False)
    if not match: return "no_atom_correspondence", 0
    for a in ref.GetAtoms():                      # 已冻结的立体中心
        rc = a.GetPropsAsDict().get("_CIPCode")
        if rc is not None:
            bc = b.GetAtomWithIdx(match[a.GetIdx()]).GetPropsAsDict().get("_CIPCode")
            if bc != rc:
                return f"STEREO_FLIP@{a.GetSymbol()}{a.GetIdx()}:{rc}->{bc}", 0
    for bd in ref.GetBonds():                     # 双键立体
        if bd.GetStereo() in (Chem.BondStereo.STEREOE, Chem.BondStereo.STEREOZ):
            b2 = b.GetBondBetweenAtoms(match[bd.GetBeginAtomIdx()], match[bd.GetEndAtomIdx()])
            if b2 is None or b2.GetStereo() != bd.GetStereo():
                return "BOND_STEREO_CHANGED", 0
    # 统计被嵌入固定下来的、原本未指定的中心
    ref_set = {match[a.GetIdx()] for a in ref.GetAtoms()
               if a.GetPropsAsDict().get("_CIPCode") is not None}
    refined = sum(1 for a in b.GetAtoms()
                  if a.GetChiralTag() != Chem.ChiralType.CHI_UNSPECIFIED
                  and a.GetIdx() not in ref_set)
    return "ok", refined

res, refined_hist, flips = collections.Counter(), collections.Counter(), []
for r in recs:
    m = Chem.MolFromMolFile(r["sdf_path"], sanitize=True, removeHs=False)
    Chem.AssignStereochemistryFrom3D(m)
    verdict, n_ref = check(r["canonical_isomeric_smiles"], m)
    res[verdict.split("@")[0]] += 1
    if verdict == "ok": refined_hist[n_ref] += 1
    else: flips.append((r["name"], verdict))

print("verdicts:", dict(res))
print("嵌入固定的未指定立体中心数 / 分子:", dict(sorted(refined_hist.items())))
for f in flips[:20]: print("  ", f)

verdicts: {'ok': 1765}
嵌入固定的未指定立体中心数 / 分子: {0: 1556, 1: 138, 2: 63, 3: 6, 4: 1, 5: 1}


In [21]:
# ── Cell 7 · 将 291 个"嵌入固定"中心按类别划分 ──
from pathlib import Path
import json, collections
from rdkit import Chem
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

OUT_DIR = Path("/mnt/hpc/home/wanghongmei/lang2/results/02_master_molecules")
recs = [json.loads(l) for l in (OUT_DIR / "identity_manifests.jsonl")
        .read_text(encoding="utf-8").splitlines() if l.strip()]

def refined_centres(ref_smi, mol3d):
    """返回 [(atom_idx, symbol, klass)]，klass ∈ {labile_N, symmetric_C, other}。"""
    ref = Chem.MolFromSmiles(ref_smi)
    Chem.AssignStereochemistry(ref, cleanIt=True, force=True)
    b = Chem.RemoveHs(Chem.Mol(mol3d))
    for a in b.GetAtoms(): a.SetAtomMapNum(0)
    Chem.AssignStereochemistry(b, cleanIt=True, force=True)
    match = b.GetSubstructMatch(ref, useChirality=False)
    frozen = {match[a.GetIdx()] for a in ref.GetAtoms()
              if a.GetPropsAsDict().get("_CIPCode") is not None}
    out = []
    for a in b.GetAtoms():
        if a.GetChiralTag() == Chem.ChiralType.CHI_UNSPECIFIED: continue
        if a.GetIdx() in frozen: continue
        deg = a.GetDegree() + a.GetTotalNumHs()
        if a.GetSymbol() == "N" and a.GetFormalCharge() > 0 and deg == 4:
            k = "labile_N"                       # 真实可互变 → 值得枚举
        elif a.GetSymbol() == "C":
            k = "symmetric_C"                    # 同一分子的假象 → 仅记录
        else:
            k = "other"                          # 需人工判断（S/P 等）
        out.append((a.GetIdx(), a.GetSymbol(), k))
    return out

per_class = collections.Counter(); mols_with = collections.Counter()
enum_extra = 0; annotated = []
for r in recs:
    m = Chem.MolFromMolFile(r["sdf_path"], sanitize=True, removeHs=False)
    Chem.AssignStereochemistryFrom3D(m)
    cs = refined_centres(r["canonical_isomeric_smiles"], m)
    for _, _, k in cs: per_class[k] += 1
    n_lab = sum(1 for _, _, k in cs if k == "labile_N")
    if cs: mols_with[";".join(sorted({k for _, _, k in cs}))] += 1
    if n_lab: enum_extra += 2 ** n_lab - 1        # 枚举所需新增分子数
    r["refined_stereocentres"] = [{"atom_idx": i, "symbol": s, "class": k} for i, s, k in cs]
    annotated.append(r)

print("centres by class:", dict(per_class))
print("molecules by class combination:", dict(mols_with))
print(f"labile-N enumeration would add {enum_extra} molecules "
      f"({len(recs)} -> {len(recs)+enum_extra}, +{100*enum_extra/len(recs):.1f}%)")

# 回写增补字段（先备份原文件）
with open(OUT_DIR / "identity_manifests_annotated.jsonl", "w", encoding="utf-8") as f:
    for r in annotated: f.write(json.dumps(r, ensure_ascii=False) + "\n")

centres by class: {'labile_N': 178, 'symmetric_C': 113}
molecules by class combination: {'labile_N': 152, 'symmetric_C': 47, 'labile_N;symmetric_C': 10}
labile-N enumeration would add 194 molecules (1765 -> 1959, +11.0%)
